[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/01_fundamentals/training_loop_from_scratch.ipynb)



# 06 . Training Loop From Scratch

## Concept — Training Loop From Scratch (revision notes)

**What it is**
- The actual code that repeatedly shows a model data, measures how wrong it
  is, and nudges its parameters to be less wrong — the process "training" refers to.
- Every high-level trainer (HuggingFace `Trainer`, PyTorch Lightning,
  `accelerate`) is ultimately a wrapper around the same loop shown here.
  Understanding it by hand means you can debug ANY of those abstractions
  when something goes wrong.

**The basic shape: two nested loops**
- **Outer loop — epochs:** one full pass through the entire training dataset.
- **Inner loop — batches:** one forward/backward/update cycle per batch,
  supplied by the `DataLoader`.
- Pseudocode:
  ```
  for epoch in range(num_epochs):
      for batch in train_loader:
          forward pass -> compute loss -> backward pass -> optimizer step
  ```

**The per-batch cycle, in order (this exact order matters)**
1. `optimizer.zero_grad()` — clear gradients from the previous step.
2. `predictions = model(inputs)` — forward pass.
3. `loss = loss_fn(predictions, targets)` — compute how wrong the predictions are.
4. `loss.backward()` — compute gradients via autograd.
5. `optimizer.step()` — update parameters using those gradients.

**train() vs eval() mode**
- Call `model.train()` before the training loop (or at the start of every
  epoch) — activates Dropout, uses batch statistics in BatchNorm.
- Call `model.eval()` before any validation/evaluation pass — deactivates
  Dropout, uses running statistics in BatchNorm.
- Forgetting to switch back and forth is one of the most common real-world
  training bugs — it silently corrupts your validation metrics or your training dynamics.

**Validation loop — the differences from training**
- No `optimizer.zero_grad()`, no `.backward()`, no `optimizer.step()` — you're
  only measuring performance, not updating weights.
- Wrap the whole thing in `with torch.no_grad():` — saves memory and time
  since no gradient graph needs to be built.
- Always call `model.eval()` first, and `model.train()` again afterward if
  you're returning to training.

**Tracking metrics over an epoch**
- Losses are usually per-batch; to report a meaningful "epoch loss" you
  average over all batches (often weighted by batch size, in case the last
  batch is smaller).
- Common pattern: accumulate `running_loss += loss.item() * batch_size`,
  then divide by the total number of samples at the end of the epoch.
- **Always `.item()` the loss before accumulating** — otherwise you keep the
  entire computation graph alive across the whole epoch (a classic memory leak).

**Checkpointing — saving the best model**
- Track a validation metric (usually loss) each epoch; whenever it improves,
  save `model.state_dict()` to disk, overwriting the previous best.
- This protects you from cases where the model overfits later in training —
  you keep the best version seen, not just the last one.

**Early stopping**
- Stop training once the validation metric hasn't improved for N consecutive
  epochs (the "patience"), instead of always running a fixed number of epochs.
- Prevents wasted compute and overfitting from training long past the point
  of diminishing returns.

**Device management inside the loop**
- The model is moved to the device ONCE, before training starts:
  `model.to(device)`.
- Every batch of data must be moved EVERY iteration, since `DataLoader`
  output stays on CPU by default: `inputs, targets = inputs.to(device), targets.to(device)`.

**What a "complete" training loop typically includes**
- Epoch loop, batch loop, train/eval mode switches, loss tracking, a
  validation pass, checkpointing the best model, optional early stopping,
  optional LR scheduler `.step()`, and some form of logging/printing progress.


### 1. The minimal single-epoch loop

**What this cell does (plain language):** the smallest possible training
loop — one pass over a toy dataset, printing the loss for each batch — so
you can see the five-step cycle (zero_grad, forward, loss, backward, step)
with nothing else added yet.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)

# toy regression problem: y = 3x + 1, plus a little noise
x = torch.linspace(-5, 5, 100).unsqueeze(1)
y = 3 * x + 1 + torch.randn_like(x) * 0.5

dataset = TensorDataset(x, y)
train_loader = DataLoader(dataset, batch_size=16, shuffle=True)

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for batch_x, batch_y in train_loader:
    optimizer.zero_grad()               # 1. clear old gradients
    predictions = model(batch_x)        # 2. forward pass
    loss = loss_fn(predictions, batch_y)  # 3. compute loss
    loss.backward()                     # 4. compute gradients
    optimizer.step()                    # 5. update parameters
    print(f"batch loss: {loss.item():.4f}")


batch loss: 86.7041
batch loss: 65.2362
batch loss: 44.6439
batch loss: 38.0624
batch loss: 16.7013
batch loss: 12.4463
batch loss: 14.4838


### 2. Multiple epochs with average epoch loss

**What this cell does (plain language):** we wrap the single-epoch loop in
an outer epoch loop, and correctly compute the AVERAGE loss for the whole
epoch (not just the last batch's loss) by accumulating `loss.item()` weighted
by batch size.


In [2]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

num_epochs = 5
for epoch in range(num_epochs):
    running_loss = 0.0
    total_samples = 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_x)
        loss = loss_fn(predictions, batch_y)
        loss.backward()
        optimizer.step()

        # accumulate with .item() - NEVER keep the raw tensor, it carries the graph
        running_loss += loss.item() * batch_x.size(0)
        total_samples += batch_x.size(0)

    epoch_loss = running_loss / total_samples
    print(f"epoch {epoch+1}/{num_epochs}: avg loss = {epoch_loss:.4f}")


epoch 1/5: avg loss = 48.6312
epoch 2/5: avg loss = 4.5287
epoch 3/5: avg loss = 0.7400
epoch 4/5: avg loss = 0.4034
epoch 5/5: avg loss = 0.3507


### 3. Adding a validation loop — train() vs eval() in practice

**What this cell does (plain language):** we split the data into train/val
sets, and add a validation pass after each training epoch — correctly
switching `model.eval()` + `torch.no_grad()` for validation, then switching
back to `model.train()` before the next training epoch.


In [3]:
from torch.utils.data import random_split

train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_subset, val_subset = random_split(dataset, [train_size, val_size], generator=torch.Generator().manual_seed(0))

train_loader = DataLoader(train_subset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_subset, batch_size=16, shuffle=False)

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(5):
    # --- training phase ---
    model.train()  # activate training-mode behavior (matters for Dropout/BatchNorm)
    running_loss, total = 0.0, 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        total += batch_x.size(0)
    train_loss = running_loss / total

    # --- validation phase ---
    model.eval()  # switch to inference-mode behavior
    val_running_loss, val_total = 0.0, 0
    with torch.no_grad():  # no gradients needed, saves memory/time
        for batch_x, batch_y in val_loader:
            loss = loss_fn(model(batch_x), batch_y)
            val_running_loss += loss.item() * batch_x.size(0)
            val_total += batch_x.size(0)
    val_loss = val_running_loss / val_total

    print(f"epoch {epoch+1}: train_loss={train_loss:.4f}, val_loss={val_loss:.4f}")


epoch 1: train_loss=74.6627, val_loss=16.0557
epoch 2: train_loss=11.9734, val_loss=3.7188
epoch 3: train_loss=3.1141, val_loss=1.7152
epoch 4: train_loss=1.6364, val_loss=1.2747
epoch 5: train_loss=1.2472, val_loss=1.0677


### 4. Adding an accuracy metric (for a classification example)

**What this cell does (plain language):** loss alone doesn't always tell
the full story for classification, so we build a small classifier, and
track accuracy alongside loss during training.


In [4]:
torch.manual_seed(0)

# toy binary classification: is x1 + x2 > 0?
x_cls = torch.randn(200, 2)
y_cls = (x_cls[:, 0] + x_cls[:, 1] > 0).long()

cls_dataset = TensorDataset(x_cls, y_cls)
cls_train, cls_val = random_split(cls_dataset, [160, 40], generator=torch.Generator().manual_seed(0))
cls_train_loader = DataLoader(cls_train, batch_size=16, shuffle=True)
cls_val_loader = DataLoader(cls_val, batch_size=16, shuffle=False)

classifier = nn.Sequential(nn.Linear(2, 8), nn.ReLU(), nn.Linear(8, 2))
optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

def compute_accuracy(logits, labels):
    predicted_classes = logits.argmax(dim=1)
    correct = (predicted_classes == labels).sum().item()
    return correct

for epoch in range(5):
    classifier.train()
    total_loss, total_correct, total_samples = 0.0, 0, 0
    for batch_x, batch_y in cls_train_loader:
        optimizer.zero_grad()
        logits = classifier(batch_x)
        loss = loss_fn(logits, batch_y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * batch_x.size(0)
        total_correct += compute_accuracy(logits, batch_y)
        total_samples += batch_x.size(0)

    train_acc = total_correct / total_samples
    print(f"epoch {epoch+1}: train_loss={total_loss/total_samples:.4f}, train_acc={train_acc:.2%}")


epoch 1: train_loss=0.4749, train_acc=91.25%
epoch 2: train_loss=0.3511, train_acc=92.50%
epoch 3: train_loss=0.2566, train_acc=93.75%
epoch 4: train_loss=0.1957, train_acc=95.62%
epoch 5: train_loss=0.1578, train_acc=96.88%


### 5. Checkpointing the best model based on validation loss

**What this cell does (plain language):** we track the best validation loss
seen so far, and every time it improves, save the model's weights to disk —
so at the end of training we have the BEST checkpoint, not just whatever the
last epoch happened to produce.


In [5]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

best_val_loss = float("inf")
checkpoint_path = "/tmp/best_model.pt"

for epoch in range(5):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    val_running_loss, val_total = 0.0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            loss = loss_fn(model(batch_x), batch_y)
            val_running_loss += loss.item() * batch_x.size(0)
            val_total += batch_x.size(0)
    val_loss = val_running_loss / val_total

    improved = val_loss < best_val_loss
    if improved:
        best_val_loss = val_loss
        torch.save(model.state_dict(), checkpoint_path)

    print(f"epoch {epoch+1}: val_loss={val_loss:.4f}" + ("  <- saved new best checkpoint" if improved else ""))

print(f"\nbest val_loss achieved: {best_val_loss:.4f}, saved to {checkpoint_path}")


epoch 1: val_loss=10.0376  <- saved new best checkpoint
epoch 2: val_loss=3.2887  <- saved new best checkpoint
epoch 3: val_loss=2.0780  <- saved new best checkpoint
epoch 4: val_loss=1.6692  <- saved new best checkpoint
epoch 5: val_loss=1.3977  <- saved new best checkpoint

best val_loss achieved: 1.3977, saved to /tmp/best_model.pt


### 6. Early stopping — stop wasting compute once progress stalls

**What this cell does (plain language):** we implement the early stopping
logic itself (the patience counter and the "has it improved" check), and
feed it a SIMULATED sequence of validation losses that plateaus partway
through — this isolates the early-stopping mechanism itself and guarantees
you can see it trigger, rather than hoping a real toy training run happens
to plateau within a fixed number of epochs (in practice, on a small convex
problem like our regression example, validation loss often keeps slowly
improving for a very long time and patience-based stopping may not fire for
many epochs - the mechanism itself is what matters here).


In [ ]:
# simulated validation losses: improving, then genuinely plateauing for 3+ epochs
simulated_val_losses = [2.0, 1.5, 1.1, 0.9, 0.85, 0.84, 0.845, 0.842, 0.843, 0.5, 0.3]

best_val_loss = float("inf")
patience = 3
epochs_without_improvement = 0

for epoch, val_loss in enumerate(simulated_val_losses):
    if val_loss < best_val_loss - 1e-4:  # small threshold avoids stopping on noise
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    print(f"epoch {epoch+1}: val_loss={val_loss:.3f}, best={best_val_loss:.3f}, "
          f"epochs_without_improvement={epochs_without_improvement}")

    if epochs_without_improvement >= patience:
        print(f"\nEarly stopping triggered at epoch {epoch+1} (no improvement for {patience} epochs)")
        print("(notice: the val_loss actually improves again at epochs 10-11 in our")
        print(" simulated list, 0.5 and 0.3 - but we never get there, because we already")
        print(" stopped at epoch 9. This is the real tradeoff of early stopping: it can")
        print(" stop before a later genuine improvement, which is exactly why `patience`")
        print(" is a tunable knob - too small risks stopping too early, too large wastes compute.)")
        break


epoch 1: val_loss=2.000, best=2.000, epochs_without_improvement=0
epoch 2: val_loss=1.500, best=1.500, epochs_without_improvement=0
epoch 3: val_loss=1.100, best=1.100, epochs_without_improvement=0
epoch 4: val_loss=0.900, best=0.900, epochs_without_improvement=0
epoch 5: val_loss=0.850, best=0.850, epochs_without_improvement=0
epoch 6: val_loss=0.840, best=0.840, epochs_without_improvement=0
epoch 7: val_loss=0.845, best=0.840, epochs_without_improvement=1
epoch 8: val_loss=0.842, best=0.840, epochs_without_improvement=2
epoch 9: val_loss=0.843, best=0.840, epochs_without_improvement=3

Early stopping triggered at epoch 9 (no improvement for 3 epochs)
(notice: the val_loss actually improves again at epochs 10-11 in our
 simulated list, 0.5 and 0.3 - but we never get there, because we already
 stopped at epoch 9. This is the real tradeoff of early stopping: it can
 stop before a later genuine improvement, which is exactly why `patience`
 is a tunable knob - too small risks stopping too

### 7. Device management — moving model and data correctly

**What this cell does (plain language):** we show the correct pattern for
device handling in a training loop: move the model ONCE before training
starts, then move every batch EVERY iteration inside the loop (since
`DataLoader` output doesn't automatically follow the model to the GPU).


In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = nn.Linear(1, 1).to(device)  # move the MODEL once, before the loop
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for epoch in range(3):
    model.train()
    running_loss, total = 0.0, 0
    for batch_x, batch_y in train_loader:
        # move EVERY batch, EVERY iteration - this is the part people forget
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)

        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        total += batch_x.size(0)

    print(f"epoch {epoch+1}: avg loss = {running_loss/total:.4f} (all computed on {device})")


Using device: cpu
epoch 1: avg loss = 23.5895 (all computed on cpu)
epoch 2: avg loss = 4.3472 (all computed on cpu)
epoch 3: avg loss = 1.4628 (all computed on cpu)


### 8. Logging a full training history

**What this cell does (plain language):** instead of just printing progress,
we store every epoch's metrics in a `history` dictionary of lists — the
standard pattern for later plotting loss curves or comparing runs.


In [8]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

history = {"train_loss": [], "val_loss": []}

for epoch in range(5):
    model.train()
    running_loss, total = 0.0, 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        total += batch_x.size(0)
    history["train_loss"].append(running_loss / total)

    model.eval()
    val_running_loss, val_total = 0.0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            loss = loss_fn(model(batch_x), batch_y)
            val_running_loss += loss.item() * batch_x.size(0)
            val_total += batch_x.size(0)
    history["val_loss"].append(val_running_loss / val_total)

print("full history dict:")
for key, values in history.items():
    print(f"  {key}: {[round(v, 4) for v in values]}")


full history dict:
  train_loss: [52.4736, 7.7504, 1.6148, 0.6877, 0.5118]
  val_loss: [10.8081, 1.9122, 0.6751, 0.4522, 0.3913]


### 9. A learning rate scheduler integrated into the loop

**What this cell does (plain language):** we add a `CosineAnnealingLR`
scheduler and call `.step()` at the correct point (once per epoch, after
the training batches for that epoch), printing the LR at each epoch to show
it changing over the course of training.


In [9]:
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
loss_fn = nn.MSELoss()

for epoch in range(5):
    model.train()
    running_loss, total = 0.0, 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * batch_x.size(0)
        total += batch_x.size(0)

    current_lr = optimizer.param_groups[0]['lr']
    print(f"epoch {epoch+1}: avg loss = {running_loss/total:.4f}, lr = {current_lr:.5f}")
    scheduler.step()  # called once per epoch, AFTER the training batches


epoch 1: avg loss = 56.5386, lr = 0.10000
epoch 2: avg loss = 0.9757, lr = 0.09045
epoch 3: avg loss = 0.3395, lr = 0.06545
epoch 4: avg loss = 0.2970, lr = 0.03455
epoch 5: avg loss = 0.2879, lr = 0.00955


### 10. Putting it all together — one reusable train_model function

**What this cell does (plain language):** we combine everything from this
notebook (train/eval switching, metric tracking, checkpointing, early
stopping, device handling, scheduler, history logging) into a single
reusable function, matching the shape of a real project's training script.


In [10]:
def train_model(model, train_loader, val_loader, loss_fn, optimizer,
                 scheduler=None, num_epochs=20, patience=5,
                 checkpoint_path="/tmp/best_model_full.pt", device=None):
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")
    epochs_without_improvement = 0

    for epoch in range(num_epochs):
        # --- train ---
        model.train()
        running_loss, total = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * batch_x.size(0)
            total += batch_x.size(0)
        train_loss = running_loss / total
        history["train_loss"].append(train_loss)

        # --- validate ---
        model.eval()
        val_running_loss, val_total = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                loss = loss_fn(model(batch_x), batch_y)
                val_running_loss += loss.item() * batch_x.size(0)
                val_total += batch_x.size(0)
        val_loss = val_running_loss / val_total
        history["val_loss"].append(val_loss)

        if scheduler is not None:
            scheduler.step()

        improved = val_loss < best_val_loss
        if improved:
            best_val_loss = val_loss
            epochs_without_improvement = 0
            torch.save(model.state_dict(), checkpoint_path)
        else:
            epochs_without_improvement += 1

        print(f"epoch {epoch+1}/{num_epochs}: train_loss={train_loss:.4f}, "
              f"val_loss={val_loss:.4f}" + ("  <- best" if improved else ""))

        if epochs_without_improvement >= patience:
            print(f"Early stopping at epoch {epoch+1}")
            break

    model.load_state_dict(torch.load(checkpoint_path))  # restore the best checkpoint
    return model, history

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=10)

trained_model, history = train_model(
    model, train_loader, val_loader, nn.MSELoss(), optimizer,
    scheduler=scheduler, num_epochs=10, patience=3,
)
print("\nfinal learned weight:", trained_model.weight.item(), "(expected ~3.0)")
print("final learned bias:", trained_model.bias.item(), "(expected ~1.0)")


epoch 1/10: train_loss=53.5959, val_loss=11.0152  <- best
epoch 2/10: train_loss=7.9293, val_loss=1.8951  <- best
epoch 3/10: train_loss=1.6049, val_loss=0.5903  <- best
epoch 4/10: train_loss=0.6274, val_loss=0.3697  <- best
epoch 5/10: train_loss=0.4461, val_loss=0.3181  <- best
epoch 6/10: train_loss=0.4010, val_loss=0.2998  <- best
epoch 7/10: train_loss=0.3838, val_loss=0.2911  <- best
epoch 8/10: train_loss=0.3755, val_loss=0.2867  <- best
epoch 9/10: train_loss=0.3719, val_loss=0.2847  <- best
epoch 10/10: train_loss=0.3703, val_loss=0.2843  <- best

final learned weight: 2.9872491359710693 (expected ~3.0)
final learned bias: 0.7201542258262634 (expected ~1.0)


## Common bugs

- **Forgot `model.eval()` before validation**
  Dropout keeps randomly zeroing activations and BatchNorm keeps using
  batch statistics during "validation," silently making your validation
  metrics noisy and unreliable. Fix: always call `model.eval()` before any
  evaluation pass, and `model.train()` again before resuming training.

- **Forgot `torch.no_grad()` during validation**
  Validation still builds a full autograd graph, wasting memory and time for
  no benefit (you're not calling `.backward()` anyway). Fix: wrap the entire
  validation loop body in `with torch.no_grad():`.

- **Accumulating `loss` instead of `loss.item()`**
  `running_loss += loss` (without `.item()`) keeps the ENTIRE computation
  graph for every batch alive for the rest of the epoch, causing memory usage
  to balloon and eventually an out-of-memory error on longer training runs.
  Fix: always accumulate `.item()`, a plain Python float, not the tensor.

- **Epoch loss computed as a simple average of per-batch losses instead of a
  weighted average**
  If the last batch is smaller (common when the dataset size doesn't divide
  evenly by batch size), a plain average over per-batch losses over-weights
  that partial batch. Fix: accumulate `loss.item() * batch_size` and divide
  by the total number of samples, not the number of batches.

- **Forgot to move each batch to the device inside the loop**
  `model.to(device)` only needs to happen once, but every batch coming out
  of the DataLoader is still on CPU by default — forgetting to move it every
  iteration raises `RuntimeError: Expected all tensors to be on the same device`.
  Fix: `inputs, targets = inputs.to(device), targets.to(device)` inside the
  loop body, every batch.

- **Checkpointing based on training loss instead of validation loss**
  Saving "the best model" using training loss can save an overfit model —
  training loss keeps improving even as the model starts memorizing rather
  than generalizing. Fix: checkpoint based on a held-out validation metric,
  not the training metric.

- **Early stopping without a minimum-improvement threshold**
  Comparing `val_loss < best_val_loss` exactly can cause the patience
  counter to reset on meaningless floating-point noise, effectively
  disabling early stopping. Fix: use a small threshold, e.g.
  `val_loss < best_val_loss - 1e-4`, so only meaningful improvements count.

- **Calling `scheduler.step()` before `optimizer.step()`, or at the wrong frequency**
  Most schedulers expect `.step()` once per epoch, AFTER the epoch's training
  batches are done; calling it per-batch (for a per-epoch scheduler) applies
  the decay schedule far too fast. Fix: check the specific scheduler's
  convention, and call it at the matching frequency in the matching place.

- **Loading a checkpoint into a model with a different architecture**
  `RuntimeError: Error(s) in loading state_dict`. Fix: make sure the
  `nn.Module` class you're loading into is defined identically to the one
  that produced the saved weights.


## Exercises

Attempt each one in the empty cell right after the question. My full solution is in the cell after that - don't peek until you've tried.

**Exercise 1 - Write a single-epoch loop from scratch.**
Given a `DataLoader` called `loader`, a model, an optimizer, and
`nn.MSELoss()`, write the five-step training cycle for one full epoch
(no printing needed, just the mechanics).

In [11]:
# Your attempt for Exercise 1 here\n

**Solution 1**

In [12]:
# Solution 1
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

# minimal self-contained setup so this cell runs on its own
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
loader = DataLoader(TensorDataset(torch.randn(20, 1), torch.randn(20, 1)), batch_size=4)

for batch_x, batch_y in loader:
    optimizer.zero_grad()
    predictions = model(batch_x)
    loss = loss_fn(predictions, batch_y)
    loss.backward()
    optimizer.step()

print("completed one epoch over the toy loader")


completed one epoch over the toy loader


**Exercise 2 - Fix the epoch-loss averaging bug.**
This code computes epoch loss incorrectly when the last batch is a
different size - identify the bug and fix it:
```python
total_loss = 0.0
num_batches = 0
for batch_x, batch_y in loader:
    loss = loss_fn(model(batch_x), batch_y)
    total_loss += loss.item()
    num_batches += 1
epoch_loss = total_loss / num_batches
```

In [13]:
# Your attempt for Exercise 2 here\n

**Solution 2**

In [14]:
# Solution 2
# Bug: averaging per-batch losses directly over-weights a smaller final
# batch (it counts as "one batch" just like all the full-size ones).
# Fix: weight each batch's loss by its actual batch size, and divide by
# the total number of SAMPLES, not the number of batches.

total_loss = 0.0
total_samples = 0
for batch_x, batch_y in train_loader:
    loss = loss_fn(model(batch_x), batch_y)
    total_loss += loss.item() * batch_x.size(0)  # weight by batch size
    total_samples += batch_x.size(0)
epoch_loss = total_loss / total_samples
print("epoch_loss computed correctly:", epoch_loss)


epoch_loss computed correctly: 98.27005462646484


**Exercise 3 - Add a validation loop.**
Given `train_loader` and `val_loader`, write a single epoch that trains on
`train_loader` and then evaluates on `val_loader`, correctly switching
`model.train()`/`model.eval()` and using `torch.no_grad()` for validation.

In [ ]:
# Your attempt for Exercise 3 here\n

**Solution 3**

In [16]:
# Solution 3
model.train()
for batch_x, batch_y in train_loader:
    optimizer.zero_grad()
    loss = loss_fn(model(batch_x), batch_y)
    loss.backward()
    optimizer.step()

model.eval()
val_loss_total, val_samples = 0.0, 0
with torch.no_grad():
    for batch_x, batch_y in val_loader:
        loss = loss_fn(model(batch_x), batch_y)
        val_loss_total += loss.item() * batch_x.size(0)
        val_samples += batch_x.size(0)
print("validation loss:", val_loss_total / val_samples)


validation loss: 12.335735702514649


**Exercise 4 - Track and print accuracy for a classifier.**
Given a classifier producing logits of shape `(batch, num_classes)` and
integer labels, write a function `epoch_accuracy(loader, model)` that
returns the overall accuracy over an entire DataLoader (not just one batch).

In [17]:
# Your attempt for Exercise 4 here\n

**Solution 4**

In [18]:
# Solution 4
def epoch_accuracy(loader, model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for batch_x, batch_y in loader:
            logits = model(batch_x)
            predicted = logits.argmax(dim=1)
            correct += (predicted == batch_y).sum().item()
            total += batch_y.size(0)
    return correct / total

# quick test with the classifier and loaders from section 4
acc = epoch_accuracy(cls_val_loader, classifier)
print("validation accuracy:", acc)


validation accuracy: 1.0


**Exercise 5 - Implement checkpointing.**
Write a training loop over 10 epochs that saves the model's `state_dict()`
to `/tmp/exercise_checkpoint.pt` only when the validation loss improves,
and prints which epochs triggered a save.

In [19]:
# Your attempt for Exercise 5 here\n

**Solution 5**

In [20]:
# Solution 5
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
best_val_loss = float("inf")

for epoch in range(10):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss_total, val_samples = 0.0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            loss = loss_fn(model(batch_x), batch_y)
            val_loss_total += loss.item() * batch_x.size(0)
            val_samples += batch_x.size(0)
    val_loss = val_loss_total / val_samples

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), "/tmp/exercise_checkpoint.pt")
        print(f"epoch {epoch+1}: NEW BEST, val_loss={val_loss:.4f}, saved checkpoint")


epoch 1: NEW BEST, val_loss=17.3293, saved checkpoint
epoch 2: NEW BEST, val_loss=4.5302, saved checkpoint
epoch 3: NEW BEST, val_loss=2.4924, saved checkpoint
epoch 4: NEW BEST, val_loss=1.9023, saved checkpoint
epoch 5: NEW BEST, val_loss=1.5770, saved checkpoint
epoch 6: NEW BEST, val_loss=1.3288, saved checkpoint
epoch 7: NEW BEST, val_loss=1.1221, saved checkpoint
epoch 8: NEW BEST, val_loss=0.9559, saved checkpoint
epoch 9: NEW BEST, val_loss=0.8182, saved checkpoint
epoch 10: NEW BEST, val_loss=0.7086, saved checkpoint


**Exercise 6 - Implement early stopping with a minimum-improvement threshold.**
Write a training loop with `patience=4` and a minimum improvement threshold
that breaks out of the loop early if validation loss hasn't meaningfully
improved for 4 consecutive epochs. (Tip: pick a threshold large enough to
matter relative to how much the loss naturally improves per epoch as
training converges - a threshold that's too tiny may never trigger.)

In [ ]:
# Your attempt for Exercise 6 here\n

**Solution 6**

In [22]:
# Solution 6
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

best_val_loss = float("inf")
patience = 4
min_improvement = 1e-2  # deliberately not tiny - real convergence curves flatten out
                        # gradually, so this threshold is what makes "improvement"
                        # stop counting as the loss curve naturally levels off
epochs_without_improvement = 0

for epoch in range(100):
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()

    model.eval()
    val_loss_total, val_samples = 0.0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            loss = loss_fn(model(batch_x), batch_y)
            val_loss_total += loss.item() * batch_x.size(0)
            val_samples += batch_x.size(0)
    val_loss = val_loss_total / val_samples

    if val_loss < best_val_loss - min_improvement:
        best_val_loss = val_loss
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= patience:
        print(f"Early stopping at epoch {epoch+1}, best_val_loss={best_val_loss:.4f}")
        break
else:
    print("Completed all epochs without early stopping")


Early stopping at epoch 31, best_val_loss=0.1985


**Exercise 7 - Spot the device bug.**
This training loop raises a `RuntimeError` partway through - identify why
and fix it:
```python
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SomeModel().to(device)
for batch_x, batch_y in loader:
    optimizer.zero_grad()
    loss = loss_fn(model(batch_x), batch_y)
    loss.backward()
    optimizer.step()
```

In [23]:
# Your attempt for Exercise 7 here\n

**Solution 7**

In [24]:
# Solution 7
# Bug: `batch_x` and `batch_y` come from the DataLoader on CPU by default,
# but the model was moved to `device` - if device is "cuda", the forward
# pass raises: RuntimeError: Expected all tensors to be on the same device.
# Fix: move each batch to the device inside the loop, every iteration.

import torch.nn as nn

class SomeModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.linear = nn.Linear(1, 1)
    def forward(self, x):
        return self.linear(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SomeModel().to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()

for batch_x, batch_y in train_loader:
    batch_x, batch_y = batch_x.to(device), batch_y.to(device)  # the fix
    optimizer.zero_grad()
    loss = loss_fn(model(batch_x), batch_y)
    loss.backward()
    optimizer.step()
print("completed without device errors")


completed without device errors


**Exercise 8 - Log per-epoch metrics into a history dict.**
Write a training loop over 5 epochs that stores `train_loss` and `val_loss`
for every epoch into a `history` dictionary of lists, and print the full
history at the end.

In [25]:
# Your attempt for Exercise 8 here\n

**Solution 8**

In [26]:
# Solution 8
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)
loss_fn = nn.MSELoss()
history = {"train_loss": [], "val_loss": []}

for epoch in range(5):
    model.train()
    total_loss, total = 0.0, 0
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * batch_x.size(0)
        total += batch_x.size(0)
    history["train_loss"].append(total_loss / total)

    model.eval()
    val_total_loss, val_total = 0.0, 0
    with torch.no_grad():
        for batch_x, batch_y in val_loader:
            loss = loss_fn(model(batch_x), batch_y)
            val_total_loss += loss.item() * batch_x.size(0)
            val_total += batch_x.size(0)
    history["val_loss"].append(val_total_loss / val_total)

print(history)


{'train_loss': [37.252892684936526, 5.387461376190186, 1.0410785794258117, 0.4447284996509552, 0.34397507309913633], 'val_loss': [7.557225227355957, 1.2189615964889526, 0.383155083656311, 0.26086204349994657, 0.23218642324209213]}


**Exercise 9 - Integrate a scheduler correctly.**
Write a training loop that uses `torch.optim.lr_scheduler.StepLR` with
`step_size=2, gamma=0.5`, calling `.step()` at the correct point in the loop,
and print the learning rate at the start of every epoch to verify it's
decaying on schedule.

In [ ]:
# Your attempt for Exercise 9 here\n

**Solution 9**

In [28]:
# Solution 9
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=2, gamma=0.5)
loss_fn = nn.MSELoss()

for epoch in range(8):
    print(f"epoch {epoch+1}: lr at start = {optimizer.param_groups[0]['lr']:.5f}")
    model.train()
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(batch_x), batch_y)
        loss.backward()
        optimizer.step()
    scheduler.step()  # once per epoch, after the training batches


epoch 1: lr at start = 0.10000
epoch 2: lr at start = 0.10000
epoch 3: lr at start = 0.05000
epoch 4: lr at start = 0.05000
epoch 5: lr at start = 0.02500
epoch 6: lr at start = 0.02500
epoch 7: lr at start = 0.01250
epoch 8: lr at start = 0.01250


**Exercise 10 - Write a complete, reusable train_model function.**
Combine train/eval switching, metric tracking, device handling, and
checkpointing (but skip early stopping and scheduling for simplicity) into
one function `simple_train(model, train_loader, val_loader, loss_fn,
optimizer, num_epochs)` that returns the trained model and a history dict.

In [ ]:
# Your attempt for Exercise 10 here\n

**Solution 10**

In [30]:
# Solution 10
def simple_train(model, train_loader, val_loader, loss_fn, optimizer, num_epochs):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    history = {"train_loss": [], "val_loss": []}
    best_val_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()
        total_loss, total = 0.0, 0
        for batch_x, batch_y in train_loader:
            batch_x, batch_y = batch_x.to(device), batch_y.to(device)
            optimizer.zero_grad()
            loss = loss_fn(model(batch_x), batch_y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * batch_x.size(0)
            total += batch_x.size(0)
        history["train_loss"].append(total_loss / total)

        model.eval()
        val_total_loss, val_total = 0.0, 0
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x, batch_y = batch_x.to(device), batch_y.to(device)
                loss = loss_fn(model(batch_x), batch_y)
                val_total_loss += loss.item() * batch_x.size(0)
                val_total += batch_x.size(0)
        val_loss = val_total_loss / val_total
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            torch.save(model.state_dict(), "/tmp/simple_train_best.pt")

        print(f"epoch {epoch+1}: train_loss={history['train_loss'][-1]:.4f}, val_loss={val_loss:.4f}")

    return model, history

test_model = nn.Linear(1, 1)
test_optimizer = torch.optim.SGD(test_model.parameters(), lr=0.01)
trained, hist = simple_train(test_model, train_loader, val_loader, nn.MSELoss(), test_optimizer, num_epochs=3)


epoch 1: train_loss=45.2029, val_loss=9.5459
epoch 2: train_loss=6.9667, val_loss=1.8152
epoch 3: train_loss=1.5603, val_loss=0.7470
